# Etiquetado semántico — LIM y CONC

Produce el Dataset A usando búsqueda semántica (FAISS) + validación con LLM.
No depende de encabezados explícitos — funciona para fragmentos con o sin título de sección.

Método estandarizado para todo el equipo.

**Requisito:** API key de Groq (gratis en console.groq.com)

## 1. Instalación

In [ ]:
!pip install -q faiss-cpu sentence-transformers groq pyarrow pandas tqdm

## 2. Montar Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Configuración

In [ ]:
# Groq API key — obtener gratis en console.groq.com
# Opcion A: pegar aqui directamente
GROQ_API_KEY = "gsk_REEMPLAZA_CON_TU_CLAVE"
# Opcion B (recomendada): usar Secrets de Colab
# from google.colab import userdata
# GROQ_API_KEY = userdata.get('GROQ_API_KEY')

GROQ_MODEL = "llama-3.3-70b-versatile"   # modelo actual de Groq (llama3-70b-8192 fue dado de baja)

INPUT_PATH      = "/content/drive/MyDrive/camilo_corpus.parquet"
OUTPUT_PATH     = "/content/drive/MyDrive/camilo_dataset_A.parquet"

META_LIM        = 2000
META_CONC       = 2000

MIN_PALABRAS    = 250
MAX_PALABRAS    = 1000
CHUNK_SIZE      = 800
CHUNK_OVERLAP   = 200
TOP_K           = 5      # candidatos por query semántica
SEM_SCORE_MIN   = 0.40   # score semántico mínimo para llamar al LLM
SCORE_MIN       = 6      # score LLM mínimo (1-10)
GROQ_SLEEP      = 2.1    # segundos entre llamadas (tier gratis: 30 req/min)
SEED            = 42

print(f"Modelo:  {GROQ_MODEL}")
print(f"Corpus:  {INPUT_PATH}")
print(f"Meta LIM:  {META_LIM}")
print(f"Meta CONC: {META_CONC}")

## 4. Imports

In [ ]:
import json
import os
import re
import time
import numpy as np
import pandas as pd
import faiss
from groq import Groq
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

groq_client = Groq(api_key=GROQ_API_KEY)

print("Cargando modelo de embeddings...")
embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
print("Listo.")

## 5. Queries semánticas por etiqueta

In [ ]:
QUERIES_LIM = [
    "una limitación de este estudio es",
    "entre las limitaciones de esta investigación",
    "este trabajo no considera",
    "queda fuera del alcance de este estudio",
    "como trabajo futuro se propone",
    "futuras investigaciones podrían explorar",
    "este enfoque tiene la restricción de",
    "no fue posible abordar en este trabajo",
    "líneas de investigación futuras incluyen",
    "requiere mayor investigación en el futuro",
    "el tamaño de la muestra limita",
    "los resultados no pueden generalizarse",
]

QUERIES_CONC = [
    "en conclusión, este estudio demuestra",
    "los resultados indican que",
    "este trabajo concluye que",
    "en resumen, los hallazgos sugieren",
    "se puede concluir que",
    "las conclusiones de este trabajo son",
    "este estudio aporta evidencia de",
    "los hallazgos principales de esta investigación",
    "en síntesis, este trabajo muestra",
    "se concluye que los resultados",
    "la principal contribución de este trabajo",
    "este artículo demuestra que",
]

QUERIES = {
    "LIM":  QUERIES_LIM,
    "CONC": QUERIES_CONC,
}

## 6. Funciones

In [ ]:
def split_into_chunks(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    words = text.split()
    chunks, start = [], 0
    step = chunk_size - overlap
    while start < len(words):
        chunks.append(" ".join(words[start:start + chunk_size]))
        start += step
    return chunks


def word_count(text):
    return len(text.split())


def expand_chunk(chunks, idx, min_w=MIN_PALABRAS, max_w=MAX_PALABRAS):
    current = chunks[idx]
    total = word_count(current)
    left, right = idx - 1, idx + 1
    while total < min_w and (left >= 0 or right < len(chunks)):
        if left >= 0:
            c = chunks[left]
            if total + word_count(c) <= max_w:
                current = c + " " + current
                total += word_count(c)
                left -= 1
            else:
                left = -1
        if total >= min_w:
            break
        if right < len(chunks):
            c = chunks[right]
            if total + word_count(c) <= max_w:
                current = current + " " + c
                total += word_count(c)
                right += 1
            else:
                right = len(chunks)
        if left < 0 and right >= len(chunks):
            break
    return current


def build_faiss_index(chunks):
    emb = embedding_model.encode(chunks, normalize_embeddings=True, show_progress_bar=False)
    index = faiss.IndexFlatIP(emb.shape[1])
    index.add(np.array(emb, dtype="float32"))
    return index


def semantic_search(index, query, top_k=TOP_K):
    q = embedding_model.encode([query], normalize_embeddings=True)
    distances, indices = index.search(np.array(q, dtype="float32"), top_k)
    return list(zip(indices[0], distances[0]))


PROMPTS = {
    "LIM": """Eres un experto en análisis del discurso científico en español.

Determina si el siguiente fragmento pertenece a una sección de LIMITACIONES o TRABAJO FUTURO.
Evalúa la función retórica: ¿reconoce restricciones del estudio, señala qué no se pudo abordar,
indica que los resultados no son generalizables, o propone trabajo futuro?

Fragmento:
\"\"\"{text}\"\"\"

Responde ÚNICAMENTE con JSON válido:
{{"es_etiqueta": true | false, "score": número entero 1-10, "razon": "una oración"}}""",

    "CONC": """Eres un experto en análisis del discurso científico en español.

Determina si el siguiente fragmento pertenece a una sección de CONCLUSIONES.
Evalúa la función retórica: ¿sintetiza los hallazgos principales, responde la pregunta de investigación,
indica las implicaciones del trabajo, o cierra la argumentación del artículo?

Fragmento:
\"\"\"{text}\"\"\"

Responde ÚNICAMENTE con JSON válido:
{{"es_etiqueta": true | false, "score": número entero 1-10, "razon": "una oración"}}""",
}


def classify_with_llm(text, etiqueta, retries=3):
    prompt = PROMPTS[etiqueta].format(text=text[:2500])
    for attempt in range(retries):
        try:
            resp = groq_client.chat.completions.create(
                model=GROQ_MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.1,
                max_tokens=120,
            )
            raw = resp.choices[0].message.content.strip()
            match = re.search(r"\{.*\}", raw, re.DOTALL)
            if match:
                return json.loads(match.group())
        except Exception:
            time.sleep(2 ** attempt)
    return None

## 7. Cargar corpus

In [ ]:
df_corpus = pd.read_parquet(INPUT_PATH)
df_corpus = df_corpus.sample(frac=1, random_state=SEED).reset_index(drop=True)
print(f"Documentos disponibles: {len(df_corpus):,}")

## 8. Pipeline por etiqueta

In [ ]:
def run_pipeline(df, etiqueta, meta, docs_excluidos=None):
    queries = QUERIES[etiqueta]
    fragmentos = []
    textos_vistos = set()

    df_iter = df.copy()
    if docs_excluidos:
        df_iter = df_iter[~df_iter["doc_id"].isin(docs_excluidos)].reset_index(drop=True)

    pbar = tqdm(df_iter.itertuples(index=False), total=len(df_iter),
                desc=f"Buscando {etiqueta}")

    for row in pbar:
        if len(fragmentos) >= meta:
            print(f"\nCuota alcanzada: {len(fragmentos)} fragmentos de {etiqueta}.")
            break

        texto = row.texto.strip()
        if word_count(texto) < MIN_PALABRAS:
            continue

        chunks = split_into_chunks(texto)
        if len(chunks) < 2:
            continue

        index = build_faiss_index(chunks)

        # Mejor candidato por score semántico (máximo entre todas las queries)
        candidatos = {}
        for query in queries:
            for idx, score in semantic_search(index, query):
                if idx not in candidatos or score > candidatos[idx]:
                    candidatos[idx] = float(score)

        # Tomar solo el top-1 con score por encima del umbral
        top = sorted(candidatos.items(), key=lambda x: -x[1])
        if not top or top[0][1] < SEM_SCORE_MIN:
            continue

        idx, sem_score = top[0]
        fragmento = expand_chunk(chunks, idx)
        wc = word_count(fragmento)
        if wc < MIN_PALABRAS or wc > MAX_PALABRAS:
            continue
        if fragmento in textos_vistos:
            continue
        textos_vistos.add(fragmento)

        resultado = classify_with_llm(fragmento, etiqueta)
        time.sleep(GROQ_SLEEP)

        if resultado and resultado.get("es_etiqueta") and resultado.get("score", 0) >= SCORE_MIN:
            fragmentos.append({
                "doc_id":        str(row.doc_id),
                "fragmento_id":  f"{row.doc_id}_{etiqueta.lower()}_{len(fragmentos):04d}",
                "texto":         fragmento,
                "etiqueta_auto": etiqueta,
                "llm_score":     resultado["score"],
                "llm_razon":     resultado.get("razon", ""),
                "sem_score":     sem_score,
                "num_palabras":  wc,
                "asignado_a":    "Camilo",
            })

        pbar.set_postfix_str(f"{etiqueta}: {len(fragmentos)}/{meta}")

    return fragmentos

## 9. Ejecutar — LIM

In [ ]:
fragmentos_lim = run_pipeline(df_corpus, "LIM", META_LIM)
print(f"LIM encontrados: {len(fragmentos_lim):,}")

## 10. Ejecutar — CONC

In [ ]:
# Excluir docs que ya aportaron LIM para maximizar cobertura
docs_con_lim = {f["doc_id"] for f in fragmentos_lim}

fragmentos_conc = run_pipeline(df_corpus, "CONC", META_CONC, docs_excluidos=docs_con_lim)
print(f"CONC encontrados: {len(fragmentos_conc):,}")

## 11. Guardar Dataset A

In [ ]:
df_a = pd.DataFrame(fragmentos_lim + fragmentos_conc)
df_a = df_a.drop_duplicates(subset=["fragmento_id"]).reset_index(drop=True)
df_a.to_parquet(OUTPUT_PATH, index=False)

mb = os.path.getsize(OUTPUT_PATH) / (1024 ** 2)
print(f"Guardado en: {OUTPUT_PATH}")
print(f"Tamaño:      {mb:.1f} MB")
print()
print(df_a["etiqueta_auto"].value_counts().to_string())
print()
print("Score LLM promedio:")
print(df_a.groupby("etiqueta_auto")["llm_score"].mean().round(2).to_string())
df_a.head(3)